# Flood vs Permanent Water Segmentation — Final Notebook

Loads the 4 trained checkpoints from `flood-checkpoints-v3`, evaluates them, runs both local-chip and GEE inference, and validates against ground truth.

**Datasets to attach in the right sidebar:**
- `yash10chawla/final-sendata` — 7-band Sen1Floods11 data
- `yash10chawla/flood-detection-src` — the .py modules
- `yash10chawla/flood-checkpoints-v3` — the 4 trained models
- `yash10chawla/last-try-key` — GEE service-account JSON (only needed for GEE inference)

## 1. Setup

In [ ]:
import os, sys, torch, numpy as np, rasterio

SRC_DIR    = '/kaggle/input/datasets/yash10chawla/flood-detection-src'
DATA_ROOT  = '/kaggle/input/datasets/yash10chawla/final-sendata'
SPLITS     = f'{DATA_ROOT}/splits'
CKPT_DIR   = '/kaggle/input/datasets/yash10chawla/flood-checkpoints-v3'
OUT_DIR    = '/kaggle/working/inference_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
!pip install -q timm albumentations rasterio einops geedim

## 2. Load the 4 trained models

In [ ]:
from model import build_model

LOSS_NAMES = ['dice', 'jaccard', 'tversky', 'focal_tversky']
models = {}
for name in LOSS_NAMES:
    ckpt = torch.load(f'{CKPT_DIR}/best_{name}.pth', map_location=device, weights_only=False)
    m = build_model(in_channels=6, pretrained=False).to(device)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    models[name] = (m, ckpt['iou'])
    print(f'  {name:14s}  val IoU={ckpt["iou"]:.4f}')

best_name, (best_model, best_iou) = max(models.items(), key=lambda kv: kv[1][1])
print(f'\nBest single model: {best_name} (val IoU={best_iou:.4f})')

## 3. Test-set evaluation — single best vs prediction ensemble

These are the numbers to put in your BTP report.

In [ ]:
from torch.utils.data import DataLoader
from dataset import build_datasets
from metrics import MetricTracker
from tqdm import tqdm

datasets = build_datasets(DATA_ROOT, SPLITS, patch_size=512)
test_loader = DataLoader(datasets['test'], batch_size=12, shuffle=False, num_workers=4, pin_memory=True)

@torch.no_grad()
def evaluate(model_or_list, loader, device):
    is_ensemble = isinstance(model_or_list, list)
    tracker = MetricTracker()
    for batch in tqdm(loader, desc='Eval', leave=False):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        if is_ensemble:
            probs = sum(torch.softmax(m(images)[0], dim=1) for m in model_or_list) / len(model_or_list)
        else:
            logits, _ = model_or_list(images)
            probs = torch.softmax(logits, dim=1)
        tracker.update(probs, labels)
    return tracker.compute()

print(f'\nSingle best model ({best_name}):')
for k, v in evaluate(best_model, test_loader, device).items():
    print(f'  {k:18s} = {v:.4f}')

print(f'\nPrediction ensemble (all 4 models):')
for k, v in evaluate([m for m, _ in models.values()], test_loader, device).items():
    print(f'  {k:18s} = {v:.4f}')

## 3.5 Test-Time Augmentation (TTA)

For each test image, run the model on **8 D4-group transforms** (4 rotations × {original, h-flip}), reverse each transform on the output, and average the softmax probabilities. Always-helps trick — no retraining, just more inference compute.

Compares: single-model + TTA vs ensemble + TTA. Pick whichever is highest.

In [ ]:
@torch.no_grad()
def evaluate_tta(model_or_list, loader, device):
    '''D4-group TTA: 8 transforms (4 rotations × {orig, h-flip}). Averages softmax,
       then takes argmax for class prediction (handled inside MetricTracker).'''
    is_ensemble = isinstance(model_or_list, list)
    if not is_ensemble:
        model_or_list = [model_or_list]
    for m in model_or_list: m.eval()

    def _apply(x, k_rot, hflip):
        if hflip: x = torch.flip(x, dims=[-1])
        if k_rot: x = torch.rot90(x, k=k_rot, dims=[-2, -1])
        return x
    def _undo(x, k_rot, hflip):
        if k_rot: x = torch.rot90(x, k=-k_rot, dims=[-2, -1])
        if hflip: x = torch.flip(x, dims=[-1])
        return x

    transforms = [(k, h) for k in range(4) for h in (False, True)]   # 8 total
    tracker = MetricTracker()
    for batch in tqdm(loader, desc='TTA Eval', leave=False):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        probs_sum, n = None, 0
        for k_rot, hflip in transforms:
            x_aug = _apply(images, k_rot, hflip)
            for m in model_or_list:
                logits, _ = m(x_aug)
                p = torch.softmax(logits, dim=1)
                p = _undo(p, k_rot, hflip)
                probs_sum = p if probs_sum is None else probs_sum + p
                n += 1
        tracker.update(probs_sum / n, labels)
    return tracker.compute()

print(f'\nSingle best model ({best_name}) + TTA (8 transforms):')
for k, v in evaluate_tta(best_model, test_loader, device).items():
    print(f'  {k:18s} = {v:.4f}')

print(f'\nEnsemble (all 4 models) + TTA (8 transforms each = 32 forward passes/batch):')
for k, v in evaluate_tta([m for m, _ in models.values()], test_loader, device).items():
    print(f'  {k:18s} = {v:.4f}')

## 4. Inference helpers (used by Sections 5 + 6)

In [ ]:
from inference import sliding_window_predict, postprocess, visualize_flood_map, save_geotiff
from dataset import normalize_band

BAND_NAMES = ['s1_vv', 's1_vh', 'dem', 'slope', 'jrc', 'hand']

def predict_from_raw_bands(bands_raw, model, device, profile=None, base_name='out', save=True):
    '''bands_raw: list of 6 (H,W) np arrays in order [VV, VH, DEM, Slope, JRC, HAND].
       Returns (class_map, flood_mask, perm_mask) and optionally writes outputs.'''
    normalized = np.stack([normalize_band(bands_raw[i], BAND_NAMES[i]) for i in range(6)], axis=-1)
    prob_map = sliding_window_predict(model, normalized, device, window=512, stride=400)
    cls, flood, perm = postprocess(prob_map, bands_raw[3], bands_raw[4])

    if save and profile is not None:
        prof = profile.copy()
        prof.update({'driver': 'GTiff', 'count': 1, 'dtype': 'uint8'})
        prof.pop('nodata', None)
        save_geotiff(cls.astype(np.uint8),   prof, f'{OUT_DIR}/{base_name}_class_map.tif')
        save_geotiff(flood.astype(np.uint8), prof, f'{OUT_DIR}/{base_name}_flood_mask.tif')
        save_geotiff(perm.astype(np.uint8),  prof, f'{OUT_DIR}/{base_name}_permanent.tif')
        png = f'{OUT_DIR}/{base_name}_flood_map.png'
        visualize_flood_map(flood, perm, png, title=f'Flood Map — {base_name}')
        print(f'flood: {100*flood.mean():.2f}%  permanent: {100*perm.mean():.2f}%  → {png}')
    return cls, flood, perm

## 5. Local-chip inference (no GEE needed)

In [ ]:
from IPython.display import Image

CHIP_ID = 'Spain_5923267'   # change to any chip in final-sendata

sources = [('S1Hand', 1), ('S1Hand', 2), ('DEM', 1),
           ('Slope', 1), ('JRCWaterHand', 1), ('HAND', 1)]
bands_raw, profile = [], None
for subdir, b in sources:
    p = f'{DATA_ROOT}/HandLabeled/{subdir}/{CHIP_ID}_{subdir}.tif'
    with rasterio.open(p) as src:
        if profile is None: profile = src.profile.copy()
        bands_raw.append(src.read(b).astype(np.float32))

_, _, _ = predict_from_raw_bands(bands_raw, best_model, device, profile, CHIP_ID)
Image(f'{OUT_DIR}/{CHIP_ID}_flood_map.png')

## 6. GEE inference — predict on any lat/lon/date

In [ ]:
import ee
from inference import FloodPredictor

# Service-account auth (Kaggle-friendly, no OAuth popup)
SERVICE_ACCOUNT_FILE = '/kaggle/input/datasets/yash10chawla/last-try-key/gee-kaggle-project-fbe0b036073d.json'
ee.Initialize(ee.ServiceAccountCredentials(None, SERVICE_ACCOUNT_FILE))

predictor = FloodPredictor(f'{CKPT_DIR}/best_{best_name}.pth')
predictor.fetcher._ee_initialized = True

# Pick any bbox + date
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -66.0, -13.7, -65.6, -13.4   # Bolivian Beni
DATE = '2018-02-15'
GEE_TIF = f'{OUT_DIR}/gee_{DATE}.tif'

predictor.fetcher.fetch(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, DATE, GEE_TIF)

with rasterio.open(GEE_TIF) as src:
    profile = src.profile.copy()
    raw     = src.read().astype(np.float32)
bands_raw = [raw[i] for i in range(6)]   # geedim adds a 7th mask band — ignore it

_, gee_flood, gee_perm = predict_from_raw_bands(bands_raw, best_model, device, profile, f'gee_{DATE}_bolivia')
Image(f'{OUT_DIR}/gee_{DATE}_bolivia_flood_map.png')

## 7. Validation 1 — against Sen1Floods11 hand-labeled ground truth

In-distribution check. Uses the chip from Section 5.

In [ ]:
with rasterio.open(f'{OUT_DIR}/{CHIP_ID}_class_map.tif') as src: pred = src.read(1)
with rasterio.open(f'{DATA_ROOT}/HandLabeled/LabelHand/{CHIP_ID}_LabelHand.tif') as src: lbl = src.read(1)
with rasterio.open(f'{DATA_ROOT}/HandLabeled/JRCWaterHand/{CHIP_ID}_JRCWaterHand.tif') as src: jrc = src.read(1)

truth = lbl.astype(np.int64).copy()
truth[(truth == 1) & (jrc >= 0.5)] = 2
valid = truth >= 0

print(f'In-distribution validation on {CHIP_ID}:')
for c, name in [(1, 'flood'), (2, 'permanent')]:
    p = pred[valid] == c
    t = truth[valid] == c
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    iou = tp / max(tp+fp+fn, 1)
    print(f'  {name:9s}: IoU={iou:.3f}  P={tp/max(tp+fp,1):.3f}  R={tp/max(tp+fn,1):.3f}  '
          f'(truth={int(t.sum()):,}  pred={int(p.sum()):,})')

## 8. Validation 2 — against JRC global ground truth (out-of-distribution)

Uses the GEE prediction from Section 6. Compares model's permanent-water output against JRC GSW.

In [ ]:
import geedim
from rasterio.warp import reproject, Resampling

# Pull JRC seasonality for the same bbox + resize to match prediction grid
region = ee.Geometry.Rectangle([LON_MIN, LAT_MIN, LON_MAX, LAT_MAX])
jrc_truth = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('seasonality').unmask(0).clip(region)
geedim.MaskedImage(jrc_truth).download(f'{OUT_DIR}/jrc_truth.tif',
    region=region.getInfo(), scale=30, crs='EPSG:4326', overwrite=True)

with rasterio.open(GEE_TIF) as src: dst_profile = src.profile.copy()
jrc_resized = np.zeros(gee_perm.shape, dtype=np.float32)
with rasterio.open(f'{OUT_DIR}/jrc_truth.tif') as src:
    reproject(source=src.read(1).astype(np.float32), destination=jrc_resized,
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=dst_profile['transform'], dst_crs=dst_profile['crs'],
              resampling=Resampling.nearest)

print('Out-of-distribution validation (Bolivia GEE scene vs JRC ground truth):')
for thresh in (1, 5):
    truth = (jrc_resized >= thresh).astype(np.uint8)
    tp = ((gee_perm == 1) & (truth == 1)).sum()
    fp = ((gee_perm == 1) & (truth == 0)).sum()
    fn = ((gee_perm == 0) & (truth == 1)).sum()
    iou = tp / max(tp+fp+fn, 1)
    label = 'training-scale (binary)' if thresh == 1 else 'strict (5+ months)'
    print(f'  JRC truth >= {thresh} ({label}): IoU={iou:.3f}  P={tp/max(tp+fp,1):.3f}  R={tp/max(tp+fn,1):.3f}')

## Done

Numbers for the BTP report come from:
- **Section 3**: `iou_mean` / `iou_flood` / `iou_permanent` for both single best and ensemble
- **Section 7**: per-chip in-distribution IoU against hand labels
- **Section 8**: out-of-distribution IoU against JRC global ground truth

PNGs in `/kaggle/working/inference_outputs/` — download from the right sidebar to embed in your report.